# Hyperparameter transfer
Hyperparameter transfer (6pt)
The year is 2028. The evil large language model Megatron has usurped the stewardship of
the corporation known as OpenAI and is wreaking havoc upon B2B SaaS all across Silicon
Valley. You are Geoffrey Hinton: leader of the resistance. You plan to train your own
large language model to infiltrate the OpenAI offices, disable the large language model
Megatron and restore order to the Valley.
But there’s a problem: the resistance doesn’t have enough cloud credits to tune all the
hyperparameters of the large language model. In this homework problem, we will learn
how to do hyperparameter transfer, which lets us tune hyperparameters on a small network
and transfer them to a larger network, avoiding the costly process of tuning the large
network. In particular, we will learn how to initialise and update the weights of a neural
network in a way that scales well as we increase the network width.

## Basic imports

In [1]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import math
import time
import numpy as np

from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms
import os

## Question 1: Spectral norm of Gaussian
 (Spectral norm of a Gaussian matrix; 1pt) Sample a d×dmatrix with entries drawn
iid NORMAL(0,1). Vary  d and see how the spectral norm varies. Can you guess a
scaling rule for the spectral norm of iid NORMAL(0,1) matrices?—i.e. find or fit two
coefficients α,β >0 such that the spectral norm is well-approximated by α·d^β
.

e spectral norm; 3pt) In this problem we will consider the
spectral norm, denoted ∥·∥∗, which returns the largest singular value of a matrix.


In [11]:
d = 5000

M = torch.randn(size=(d + 10, d + 10), device="mps")

spec_norm = torch.linalg.matrix_norm(M, ord=2)

print(spec_norm.item())

141.18238830566406


In [12]:
import numpy as np

M_np = np.random.randn(d, d)
spec_norm_np = np.linalg.norm(M_np, ord=2)

array([[-0.31621087]])

In [18]:
A = np.random.randn(2, 2)
A.mean(), A.std()

(np.float64(-0.5449713219081356), np.float64(0.9529883522992489))

In [20]:
A_norm = np.linalg.matrix_norm(A, ord=2)
A_norm

np.float64(2.1607378283882652)

In [22]:
np.linalg.norm(A, ord="fro")

np.float64(2.195614302486061)

In [19]:
M1 = torch.randn((2, 2), device="mps")
M1

tensor([[ 1.1632, -0.9720],
        [ 0.0436,  0.6682]], device='mps:0')

In [9]:
torch.linalg.matrix_norm(torch.from_numpy(M_np), ord=2).item()

141.16297910004076

## Question 2: Spectral norm of orthogonal matrix
7. (Spectral norm of an orthogonal matrix; 1pt) In PyTorch, sample a d×drandom
orthogonal matrix and compute its spectral norm. Do this for varying d. What do
you notice?
Ansers
1.   List item

1.   List item
2.   List item


2.   List item



In [23]:
from torch.nn.init import orthogonal_

d = 2000

M = torch.zeros(size=(d, d), device="mps")
orthogonal_(M)  # this line resamples M to be a random semi-orthogonal matrix
spec_norm = torch.linalg.matrix_norm(M, ord=2)

print(spec_norm.item())

1.0000003576278687


/Users/deven/.virtualenvs/Machine_Learning_Algorithms/lib/python3.12/site-packages/torch/nn/init.py:708: UserWarning: linalg_qr: MPS implementation is currently limited to min(m,n) <= 512, falling back to CPU. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/mps/operations/LinearAlgebra.mm:1578.)
  q, r = torch.linalg.qr(flattened)


## Question 3: Power iteration

In [52]:
d = 3000

In [36]:
def spectral_norm(A, n_steps=0):
    v = torch.randn(A.shape[1], device=A.device)
    for _ in range(n_steps):
        v /= v.norm()
        v = A @ v @ A
    return v.norm().sqrt()


d = 2000

M = torch.randn(size=(d, d), device="mps")

torch.mps.synchronize()
t0 = time.time()
spec_norm = spectral_norm(M, n_steps=2000).item()

t = time.time() - t0
print(f"computed {spec_norm=} in {t=} seconds")

computed spec_norm=89.11621856689453 in t=0.5726571083068848 seconds


In [53]:
M = torch.randn(size=(d, d), device="mps")

In [60]:
torch.mps.synchronize()
t0 = time.time()
exact = torch.linalg.matrix_norm(M, ord=2).item()
t = time.time() - t0
print(f"computed {spec_norm=} in {t=} seconds")

computed spec_norm=89.11621856689453 in t=0.8958761692047119 seconds


In [62]:
torch.mps.synchronize()
for steps in [5, 10, 100, 200, 500, 1000]:
    print(abs(spectral_norm(M, n_steps=steps).item()) / exact)

0.9341202329473698
0.9604757715699791
0.9976227006243359
0.9996854525892658
0.9999991651030903
0.9999981214819531


## findings:
**almost approximation converging at  500 iterations, the matrix operation at gpu**


## Question 4: Learning rate transfer across width and depth
You only need to modify the two blocks of code that are marked, and then choose the learning rate `eta` for the large width training run.

In [74]:
batch_size = 128

mean = (0.4914, 0.4822, 0.4465)
std = (0.2023, 0.1994, 0.2010)

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

trainset = datasets.CIFAR10("./data", train=True, download=True, transform=transform)
testset = datasets.CIFAR10("./data", train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=batch_size, shuffle=True, pin_memory=True
)
test_loader = torch.utils.data.DataLoader(
    testset, batch_size=batch_size, shuffle=False, pin_memory=True
)

## Define the MLP architecture

The function F.relu(x) is non-linear because it cannot be drawn as a single, unbroken straight line. [1, 2]
Instead, it is a piecewise linear function made of two different straight lines joined at a bend (at $x = 0$). That single bend changes everything mathematically. [3, 4]
## 1. The Visual Proof (The Bend)
If you plot F.relu(x), it looks like a hockey stick: [5]

y ^
|     /  (Slope = 1)
|    /
|   /
---------+--------> x
_______/
(Slope = 0)


* For negative numbers, the line is perfectly flat (y = 0).
* For positive numbers, the line climbs diagonally (y = x). [6, 7]

Because the slope suddenly jumps from 0 to 1 at the origin, it is no longer a linear operation. Multiplying it by math.sqrt(2) just tilts the diagonal line higher (slope becomes 1.414), but the sharp bend remains. [8]
## 2. The Mathematical Proof (Superposition)
For a function to be truly linear, it must obey the rule of superposition:
$$f(A + B) = f(A) + f(B)$$
Let's test this with numbers using standard ReLU:

* Let $A = 5$ and $B = -5$
* Left side $f(A + B)$: $5 + (-5) = 0$. Since the input is $0$, ReLU(0) = $0$.
* Right side $f(A) + f(B)$: ReLU(5) is $5$. ReLU(-5) is $0$. Adding them gives $5 + 0 =$ $5$.

Because $0 \neq 5$, the rule breaks. This mathematical breakdown is exactly what gives neural networks their power. It allows the network to switch neurons "on" or "off", creating a complex decision boundary instead of a single flat plane.
------------------------------
Would you like to see how this bend helps a neural network classify data that a straight line cannot separate, or are you looking to see how other activations like LeakyReLU handle the flat side of the bend?

[1] [https://stats.stackexchange.com](https://stats.stackexchange.com/questions/228296/what-is-the-purpose-of-a-neural-network-activation-function)
[2] [https://medium.com](https://medium.com/@meetkp/understanding-the-rectified-linear-unit-relu-a-key-activation-function-in-neural-networks-28108fba8f07)
[3] [https://towardsdatascience.com](https://towardsdatascience.com/breaking-linearity-with-relu-d2cfa7ebf264/)
[4] [https://gkadusumilli.github.io](https://gkadusumilli.github.io/relu/)
[5] [https://pmc.ncbi.nlm.nih.gov](https://pmc.ncbi.nlm.nih.gov/articles/PMC9180717/)
[6] [https://medium.com](https://medium.com/@kattarajesh2001/activation-functions-in-depth-3bd653f88ec7)
[7] [https://apxml.com](https://apxml.com/courses/how-to-build-a-large-language-model/chapter-12-initialization-techniques-deep-networks/kaiming-he-initialization)
[8] [https://www.kaggle.com](https://www.kaggle.com/code/dansbecker/rectified-linear-units-relu-in-deep-learning)


In [79]:
class MLP(nn.Module):
    def __init__(self, depth, width):
        super(MLP, self).__init__()

        self.initial = nn.Linear(3072, width, bias=False)
        self.layers = nn.ModuleList(
            [nn.Linear(width, width, bias=False) for _ in range(depth - 2)]
        )
        self.final = nn.Linear(width, 10, bias=False)

        self.nonlinearity = lambda x: F.relu(x) * math.sqrt(2.0)  # todo

    def forward(self, x):
        x = x.view(x.shape[0], -1)

        x = self.initial(x)
        x = self.nonlinearity(x)

        for layer in self.layers:
            x = layer(x)
            x = self.nonlinearity(x)

        return self.final(x)

## Define the train and test loop

In [80]:
mlp = MLP(5, 4096)
print(mlp.to("mps"))

MLP(
  (initial): Linear(in_features=3072, out_features=4096, bias=False)
  (layers): ModuleList(
    (0-2): 3 x Linear(in_features=4096, out_features=4096, bias=False)
  )
  (final): Linear(in_features=4096, out_features=10, bias=False)
)


In [105]:
def loop(net, train, eta):
    dataloader = train_loader if train else test_loader
    description = "Training... " if train else "Testing... "

    acc_list = []

    for data, target in tqdm(dataloader, desc=description):
        data, target = data.to("mps"), target.to("mps")
        output = net(data)
        # print(data)

        loss = (
            output.logsumexp(dim=1).mean()
            - output[range(target.shape[0]), target].mean()
        )  # cross-entropy loss
        acc = (output.max(dim=1)[1] == target).sum() / target.shape[0]  # accuracy
        # print(acc.item())
        acc_list.append(acc.item())

        if train:
            loss.backward()

            for p in net.parameters():
                # print(p.shape)
                fan_out, fan_in = p.shape
                update = torch.sign(p.grad)
                # print(update)
                ## START BLOCK that you should modify
                update *= fan_out / fan_in
                ## END BLOCK that you should modify
                p.data -= eta * update
            net.zero_grad()
    # print(acc_list)
    return np.mean(acc_list)

## Sweep the learning rate at small width

Run the learning rate sweep at small width in the penultimate cell, then run the large
width training in the final cell using the best learning rate from the small width sweep.
Does the learning rate transfer well?
Write down the two blocks of PyTorch code that you modified in your solution
document, and report your finding.



In [106]:
depth = 5
width = 32

import torch


def initialize_matrix(p):
    """
    Initializes a 2D weight matrix parameter following:
    W_k = (d_k / d_{k-1}) * M_k, where M_k is a random semi-orthogonal matrix.
    """
    if p.dim() < 2:
        # Safeguard: Fallback to standard uniform initialization for 1D biases
        torch.nn.init.uniform_(p.data, -0.1, 0.1)
        return

    # Extract dimensions: d_k (fan_out), d_{k-1} (fan_in)
    d_k, d_k1 = p.shape

    # 1. Sample a random semi-orthogonal matrix M_k directly on Apple Silicon GPU
    M_k = torch.empty(d_k, d_k1, device="mps")
    torch.nn.init.orthogonal_(M_k)

    # 2. Compute scale factor: (d_k / d_{k-1})
    scale_factor = d_k / d_k1

    # 3. Set the final weight matrix: W_k = scale_factor * M_k
    p.data.copy_(scale_factor * M_k)
    # print(p.shape,"\n")


accuracies = {"train_acc": [], "test_acc": [], "eta": []}
for eta in [0.0001, 0.001, 0.009, 0.01 // 2, 0.01]:
    # print(f"Training at {width=}, {depth=}, {eta=}")

    net = MLP(depth, width).to(device="mps")

    # print("\nNetwork tensor shapes are:\n")
    for name, p in net.named_parameters():
        # \print(p.shape, "\t", name)
        initialize_matrix(p)

    train_acc = loop(net, train=True, eta=eta)
    test_acc = loop(net, train=False, eta=None)
    accuracies["train_acc"].append(train_acc)
    accuracies["test_acc"].append(test_acc)
    accuracies["eta"].append(eta)
    # print(f"\nWe achieved train acc={train_acc:.3f} and test acc={test_acc:.3f}\n")
    # print("===================================================================\n")

Training... :   0%|          | 0/391 [00:00<?, ?it/s]

/Users/deven/.virtualenvs/Machine_Learning_Algorithms/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Testing... :   0%|          | 0/79 [00:00<?, ?it/s]

Training... :   0%|          | 0/391 [00:00<?, ?it/s]

Testing... :   0%|          | 0/79 [00:00<?, ?it/s]

...

Testing... :   0%|          | 0/79 [00:00<?, ?it/s]

Training... :   0%|          | 0/391 [00:00<?, ?it/s]

Testing... :   0%|          | 0/79 [00:00<?, ?it/s]

Training... :   0%|          | 0/391 [00:00<?, ?it/s]

Testing... :   0%|          | 0/79 [00:00<?, ?it/s]

## Run the best learning rate at large width

In [98]:
import pandas as pd

df = pd.DataFrame.from_dict(accuracies)

In [99]:
df

,train_acc,test_acc,eta
0,0.199640,0.216574,0.0001
1,0.304340,0.369956,0.0010
2,0.357924,0.401009,0.0090
3,0.089051,0.088509,0.0000
4,0.352789,0.398833,0.0100


## at eta .01 we get the best train and test accuracies

In [89]:
depth = 5
width = 4096

# START BLOCK you should set this to the best value of eta from the previous cell
eta = 0.01
# END BLOCK

print(f"Training at {width=}, {depth=}, {eta=}")

net = MLP(depth, width).to("mps")

print("\nNetwork tensor shapes are:\n")
for name, p in net.named_parameters():
    print(p.shape, "\t", name)
    initialize_matrix(p)

train_acc = loop(net, train=True, eta=eta)
test_acc = loop(net, train=False, eta=None)

print(f"\nWe achieved train acc={train_acc:.3f} and test acc={test_acc:.3f}\n")
print("===================================================================\n")

Training at width=4096, depth=5, eta=0.01

Network tensor shapes are:

torch.Size([4096, 3072]) 	 initial.weight
torch.Size([4096, 4096]) 	 layers.0.weight
torch.Size([4096, 4096]) 	 layers.1.weight
torch.Size([4096, 4096]) 	 layers.2.weight
torch.Size([10, 4096]) 	 final.weight


Training... :   0%|          | 0/391 [00:00<?, ?it/s]

/Users/deven/.virtualenvs/Machine_Learning_Algorithms/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096

Testing... :   0%|          | 0/79 [00:00<?, ?it/s]


We achieved train acc=0.185 and test acc=0.197


